In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [6]:
gpus = pd.read_csv('nvidia_gpus.csv')
gpus['TFLOPS/Watts'] = gpus['TFLOPS']/gpus['Watts']
gpus

,GPU,Precision,TFLOPS,Watts,Launch date,Type,TFLOPS/Watts
0,GeForce GTX 580,FP32,1.58,244,09/11/2010,Desktop,0.006475
1,GeForce GTX 590,FP32,2.49,365,24/03/2011,Desktop,0.006822
2,GeForce GTX 680,FP32,3.09,195,22/03/2012,Desktop,0.015846
3,GeForce GTX 690,FP32,5.62,300,29/04/2012,Desktop,0.018733
4,GeForce GTX 780,FP32,4.16,250,23/05/2013,Desktop,0.016640
5,GeForce GTX 780 TI,FP32,5.35,250,07/11/2013,Desktop,0.021400
6,GeForce GTX Titan Black,FP32,5.65,250,18/02/2014,Desktop,0.022600
7,GeForce GTX Titan Z,FP32,8.12,375,28/05/2014,Desktop,0.021653
8,GeForce GTX 980,FP32,4.98,165,18/09/2014,Desktop,0.030182
9,GeForce GTX 980 Ti,FP32,6.06,250,02/06/2015,Desktop,0.024240


In [56]:
### One epoch
C0_T = 6
C0_t = 1.5

params = np.linspace(1e7, 1e10, 50)   # 10 million to 100 billion parameters (linear spacing)
tokens_T = np.linspace(1e3, 1e5, 50)   # 100 million to 1 trillion tokens (linear spacing)
tokens_t = np.linspace(1e2,3.2e4,50)
P, T_T = np.meshgrid(params, tokens_T)
P, T_t = np.meshgrid(params, tokens_t)

TFLOPs_train = (C0_T * P * T_T) / 1e12
TFLOPs_inf   = (C0_t * P * T_t) / 1e12

In [62]:
fig = go.Figure(data=[go.Surface(x=P, y=T_T, z=TFLOPs_train)])
fig.update_layout(
    title='One-epoch training TFLOPs',
    scene=dict(
        xaxis=dict(title='Number of Parameters (N)'),
        yaxis=dict(title='Number of Tokens (D)'),
        zaxis=dict(title='TFLOPs'),
    ),
    width=1000,   
    height=800
)
fig.show()

In [ ]:
fig = go.Figure(data=[go.Surface(x=P, y=T_t, z=TFLOPs_inf)])
fig.update_layout(
    title='Inference TFLOPs',
    scene=dict(
        xaxis=dict(title='Number of Parameters (N)'),
        yaxis=dict(title='Number of Tokens (D)'),
        zaxis=dict(title='TFLOPs'),
    ),
    width=1000,   
    height=800
)
fig.show()

In [40]:
# Assuming T = 3.2e5
T0 = 3.2e5
TFLOPs_train = (C0_T * params * T0) / 1e12
TFLOPs_inf   = (C0_t * params * T0) / 1e12

kWh_T = (TFLOPs_train[:,None] @ gpus['TFLOPS/Watts'].values[None,:]**-1) * 1e-3 * 3.6e-3
kWh_t = (TFLOPs_inf[:,None] @ gpus['TFLOPS/Watts'].values[None,:]**-1) * 1e-3 * 3.6e-3

In [65]:
fig = go.Figure()
for i,gpu in enumerate(gpus['GPU'].values):
    fig.add_trace(go.Scatter(
        x=params, 
        y=kWh_T[:,i], 
        mode='lines',
        name=gpu,
        line=dict(width=1)
    ))

fig.update_layout(
    title="One-epoch Training Electrical Consumption per GPU (3.6k tokens)",
    xaxis_title="Number of Parameters (N)",
    yaxis_title="kWh",
    legend=dict(traceorder="grouped", orientation="v", x=1.02, y=1),
    width=1000,   
    height=800
)
fig.show()

In [66]:
fig = go.Figure()
for i,gpu in enumerate(gpus['GPU'].values):
    fig.add_trace(go.Scatter(
        x=params, 
        y=kWh_t[:,i], 
        mode='lines',
        name=gpu,
        line=dict(width=1)
    ))

fig.update_layout(
    title="Inference Electrical Consumption per GPU (3.6k tokens)",
    xaxis_title="Number of Parameters (N)",
    yaxis_title="kWh",
    legend=dict(traceorder="grouped", orientation="v", x=1.02, y=1),
    width=1000,   
    height=800
)
fig.show()